# 🏆 Churn Prediction: Master Production Pipeline (V5.7 - SHAP+)
**Fix**: Synchronized SHAP visualization for all model types.

In [ ]:
# 1. Setup
!pip install opendatasets mlflow xgboost shap python-dotenv seaborn dagshub -q

import torch
HAS_GPU = torch.cuda.is_available()

import opendatasets as od
import os, pandas as pd, numpy as np, mlflow, mlflow.sklearn, matplotlib.pyplot as plt, seaborn as sns, shap, dagshub
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import recall_score, classification_report, accuracy_score, precision_score, f1_score
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

mlflow.sklearn.autolog(log_models=False)
dagshub.init(repo_owner="nhannhb92", repo_name="msa24-ddm501-group6-final-project", mlflow=True)

In [ ]:
# 2. Data Loading
def clean_dataset(df):
    df.columns = [col.lower().replace(' ', '_') for col in df.columns]
    df = df.drop(columns=[c for c in ['customerid', 'last_interaction'] if c in df.columns])
    if 'age' in df.columns: df = df[(df['age'] >= 18) & (df['age'] <= 120)]
    for col in [c for c in ['total_spend', 'totalcharges'] if c in df.columns]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df.dropna(subset=['churn'])

if not os.path.exists("customer-churn-dataset"):
    od.download("https://www.kaggle.com/datasets/muhammadshahidazeem/customer-churn-dataset")

df_raw = clean_dataset(pd.read_csv("customer-churn-dataset/customer_churn_dataset-training-master.csv"))
df_test_final = clean_dataset(pd.read_csv("customer-churn-dataset/customer_churn_dataset-testing-master.csv"))
df_train, df_val = train_test_split(df_raw, test_size=0.15, random_state=42, stratify=df_raw['churn'])

print(f"✅ Done: Train({len(df_train)}), Val({len(df_val)}), Test({len(df_test_final)})")

In [ ]:
# 3. Unified Training Engine with Better SHAP
def run_master_v57(df_tr, df_va, df_te, model_type):
    X_tr, y_tr = df_tr.drop(columns=['churn']), df_tr['churn']
    X_va, y_va = df_va.drop(columns=['churn']), df_va['churn']
    X_te, y_te = df_te.drop(columns=['churn']), df_te['churn']
    
    preprocessor = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), X_tr.select_dtypes(include=[np.number]).columns.tolist()),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), X_tr.select_dtypes(include=['object']).columns.tolist())
    ])
    
    preprocessor.fit(X_tr)
    X_va_t = preprocessor.transform(X_va)
    
    if model_type == "xgboost":
        clf = XGBClassifier(scale_pos_weight=(y_tr==0).sum()/(y_tr==1).sum(), device='cuda' if HAS_GPU else 'cpu', early_stopping_rounds=10)
        f_params = {"clf__eval_set": [(X_va_t, y_va)], "clf__verbose": False}
        m_params = {'clf__max_depth': [3, 5], 'clf__learning_rate': [0.1]}
    elif model_type == "random_forest":
        clf = RandomForestClassifier(class_weight='balanced', random_state=42)
        f_params = {}; m_params = {'clf__max_depth': [10, 20]}
    else:
        clf = LogisticRegression(class_weight='balanced', max_iter=2000)
        f_params = {}; m_params = {'clf__C': [0.1, 1.0]}
        
    mlflow.set_experiment("Churn_Shap_Review")
    with mlflow.start_run(run_name=f"NB_V57_{model_type.upper()}"):
        pipe = Pipeline([('pre', preprocessor), ('clf', clf)])
        search = RandomizedSearchCV(pipe, m_params, n_iter=2, cv=2, scoring='recall', verbose=1)
        search.fit(X_tr, y_tr, **f_params)
        model = search.best_estimator_
        
        # Result logging
        y_pred = model.predict(X_te)
        print(f"\n📢 Test Results ({model_type.upper()}):")
        print(classification_report(y_te, y_pred))
        
        # --- 🔍 IMPROVED SHAP ---
        try:
            # Increase sample to 200 for better density
            sample = X_te.sample(min(200, len(X_te)))
            X_t = pd.DataFrame(model.named_steps['pre'].transform(sample), columns=model.named_steps['pre'].get_feature_names_out())
            
            # Use specialized explainer with correct background for density
            if model_type == "logistic_regression":
                # Use X_t itself as background for the linear explainer to show more fields
                explainer = shap.Explainer(model.named_steps['clf'], X_t)
            else:
                explainer = shap.Explainer(model.named_steps['clf'])
                
            # Compute and Plot
            shap_values = explainer(X_t)
            plt.figure(figsize=(10, 8))
            shap.summary_plot(shap_values, X_t, show=False, max_display=20, plot_size=None)
            plt.title(f"Explaining {model_type.upper()}")
            plt.show()
        except Exception as e: 
            print(f"⚠️ SHAP Error: {e}")

        from mlflow.models import infer_signature
        mlflow.sklearn.log_model(model, "model", registered_model_name=f"CustomerChurnModel_{model_type}", signature=infer_signature(X_tr, model.predict(X_tr)))

for m in ["xgboost", "random_forest", "logistic_regression"]:
    run_master_v57(df_train, df_val, df_test_final, m)